## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 3: Matrix Factorization for Recommender Systems
## Task 6: Implementing Matrix Factorization with SGD (Funk SVD)

Implement the same algorithm that the Surprise library's SVD uses under the hood:
Funk SVD — a gradient-descent-based matrix factorization with user/item biases and L2 regularization.

Prediction formula: r_hat(u,i) = mu + b_u + b_i + p_u^T * q_i

Where:
- mu = global mean rating
- b_u = user bias
- b_i = item bias
- p_u = user latent factor vector
- q_i = item latent factor vector

We train via SGD, tune hyperparameters, evaluate with RMSE/Precision@K/Recall@K, and compare with the manual SVD from Task 5.

Dataset: MovieLens ml-latest-small

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'scikit-learn', 'scipy', '-q'])

import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
print('All imports successful!')

All imports successful!


### Step 1: Load Dataset

In [3]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'

if not os.path.exists(extract_dir):
    print('Downloading dataset...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print('Downloaded and extracted.')
else:
    print('Dataset already exists.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings_df = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f'Movies: {len(movies)}, Ratings: {len(ratings_df)}, Users: {ratings_df["userId"].nunique()}')

Dataset already exists.
Movies: 9742, Ratings: 100836, Users: 610


### Step 2: Train/Test Split

In [4]:
train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

Train: 80668, Test: 20168


### Step 3: Implement Funk SVD (SGD-based Matrix Factorization)

This is the same algorithm that the Surprise library SVD class implements:
- Initialize user/item latent factors and biases randomly
- For each epoch, iterate over all observed ratings
- Compute prediction error and update parameters via SGD
- Apply L2 regularization to prevent overfitting

In [5]:
class FunkSVD:
    """
    Funk SVD - SGD-based matrix factorization with biases.
    Equivalent to the Surprise library SVD algorithm.
    pred(u, i) = mu + b_u + b_i + p_u . q_i
    """

    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02, random_state=42):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, train_data):
        rng = np.random.RandomState(self.random_state)

        self.user_ids = train_data['userId'].unique()
        self.item_ids = train_data['movieId'].unique()
        self.user_to_idx = {uid: i for i, uid in enumerate(self.user_ids)}
        self.item_to_idx = {iid: i for i, iid in enumerate(self.item_ids)}

        n_users = len(self.user_ids)
        n_items = len(self.item_ids)

        self.mu = train_data['rating'].mean()
        self.b_u = np.zeros(n_users)
        self.b_i = np.zeros(n_items)
        self.P = rng.normal(0, 0.1, (n_users, self.n_factors))
        self.Q = rng.normal(0, 0.1, (n_items, self.n_factors))

        users = train_data['userId'].map(self.user_to_idx).values
        items = train_data['movieId'].map(self.item_to_idx).values
        rat = train_data['rating'].values

        for epoch in range(self.n_epochs):
            indices = rng.permutation(len(rat))
            total_loss = 0.0
            for idx in indices:
                u, i, r = users[idx], items[idx], rat[idx]
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                err = r - pred
                total_loss += err ** 2
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])
                P_u = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * P_u - self.reg * self.Q[i])
            rmse = np.sqrt(total_loss / len(rat))
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f'  Epoch {epoch+1:>3d}/{self.n_epochs} - Train RMSE: {rmse:.4f}')
        return self

    def predict(self, user_id, movie_id):
        if user_id in self.user_to_idx and movie_id in self.item_to_idx:
            u = self.user_to_idx[user_id]
            i = self.item_to_idx[movie_id]
            pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
        elif user_id in self.user_to_idx:
            u = self.user_to_idx[user_id]
            pred = self.mu + self.b_u[u]
        elif movie_id in self.item_to_idx:
            i = self.item_to_idx[movie_id]
            pred = self.mu + self.b_i[i]
        else:
            pred = self.mu
        return np.clip(pred, 0.5, 5.0)

    def predict_batch(self, test_data):
        return np.array([self.predict(row['userId'], row['movieId']) for _, row in test_data.iterrows()])

print('FunkSVD class defined.')

FunkSVD class defined.


### Step 4: Train Default Model

In [6]:
print('Training default Funk SVD (n_factors=50, lr=0.005, reg=0.02, epochs=20)...\n')
model_default = FunkSVD(n_factors=50, n_epochs=20, lr=0.005, reg=0.02)
model_default.fit(train_df)

test_preds = model_default.predict_batch(test_df)
rmse_default = np.sqrt(mean_squared_error(test_df['rating'], test_preds))
print(f'\nDefault Model - Test RMSE: {rmse_default:.4f}')

Training default Funk SVD (n_factors=50, lr=0.005, reg=0.02, epochs=20)...

  Epoch   1/20 - Train RMSE: 0.9647
  Epoch   5/20 - Train RMSE: 0.8602
  Epoch  10/20 - Train RMSE: 0.8175
  Epoch  15/20 - Train RMSE: 0.7763
  Epoch  20/20 - Train RMSE: 0.7248

Default Model - Test RMSE: 0.8812


### Step 5: Hyperparameter Tuning

In [7]:
train_tune, val_tune = train_test_split(train_df, test_size=0.2, random_state=42)

param_grid = [
    {'n_factors': 20,  'lr': 0.005, 'reg': 0.02, 'n_epochs': 30},
    {'n_factors': 50,  'lr': 0.005, 'reg': 0.02, 'n_epochs': 30},
    {'n_factors': 100, 'lr': 0.005, 'reg': 0.02, 'n_epochs': 30},
    {'n_factors': 50,  'lr': 0.01,  'reg': 0.02, 'n_epochs': 20},
    {'n_factors': 50,  'lr': 0.005, 'reg': 0.1,  'n_epochs': 30},
    {'n_factors': 50,  'lr': 0.01,  'reg': 0.1,  'n_epochs': 20},
]

print(f'{"n_factors":>10s} | {"lr":>6s} | {"reg":>5s} | {"epochs":>6s} | {"Val RMSE":>9s}')
print('-' * 50)

best_rmse = float('inf')
best_params = None

import io, contextlib
for params in param_grid:
    model = FunkSVD(**params, random_state=42)
    with contextlib.redirect_stdout(io.StringIO()):
        model.fit(train_tune)
    val_preds = model.predict_batch(val_tune)
    val_rmse = np.sqrt(mean_squared_error(val_tune['rating'], val_preds))
    print(f'{params["n_factors"]:>10d} | {params["lr"]:>6.3f} | {params["reg"]:>5.2f} | {params["n_epochs"]:>6d} | {val_rmse:>9.4f}')
    if val_rmse < best_rmse:
        best_rmse = val_rmse
        best_params = params

print(f'\nBest params: {best_params}')
print(f'Best validation RMSE: {best_rmse:.4f}')

 n_factors |     lr |   reg | epochs |  Val RMSE
--------------------------------------------------
        20 |  0.005 |  0.02 |     30 |    0.8672
        50 |  0.005 |  0.02 |     30 |    0.8724
       100 |  0.005 |  0.02 |     30 |    0.8799
        50 |  0.010 |  0.02 |     20 |    0.8795
        50 |  0.005 |  0.10 |     30 |    0.8633
        50 |  0.010 |  0.10 |     20 |    0.8613

Best params: {'n_factors': 50, 'lr': 0.01, 'reg': 0.1, 'n_epochs': 20}
Best validation RMSE: 0.8613


In [8]:
print(f'Training tuned model with best params on full train set...\n')
model_tuned = FunkSVD(**best_params, random_state=42)
model_tuned.fit(train_df)

test_preds_tuned = model_tuned.predict_batch(test_df)
rmse_tuned = np.sqrt(mean_squared_error(test_df['rating'], test_preds_tuned))
print(f'\nTuned Model - Test RMSE: {rmse_tuned:.4f}')

Training tuned model with best params on full train set...

  Epoch   1/20 - Train RMSE: 0.9438
  Epoch   5/20 - Train RMSE: 0.8445
  Epoch  10/20 - Train RMSE: 0.8133
  Epoch  15/20 - Train RMSE: 0.7884
  Epoch  20/20 - Train RMSE: 0.7606

Tuned Model - Test RMSE: 0.8740


### Step 6: Generate Top-N Recommendations

In [9]:
def recommend_funk_svd(user_id, model, train_data, movies_df, top_n=10):
    rated = set(train_data[train_data['userId'] == user_id]['movieId'])
    all_movies = movies_df['movieId'].unique()
    predictions = []
    for mid in all_movies:
        if mid not in rated:
            pred = model.predict(user_id, mid)
            predictions.append((mid, pred))
    predictions.sort(key=lambda x: x[1], reverse=True)
    top = predictions[:top_n]
    results = pd.DataFrame({
        'movieId': [m for m, _ in top],
        'Predicted Rating': [round(r, 2) for _, r in top]
    }).merge(movies_df[['movieId', 'title', 'genres']], on='movieId')
    return results[['title', 'genres', 'Predicted Rating']]

for uid in [1, 5, 100]:
    print('=' * 70)
    print(f'Top 10 Funk SVD Recommendations for User {uid}')
    print('=' * 70)
    recs = recommend_funk_svd(uid, model_tuned, train_df, movies, top_n=10)
    print(recs.to_string(index=False))
    print()

Top 10 Funk SVD Recommendations for User 1
                           title                   genres  Predicted Rating
Shawshank Redemption, The (1994)              Crime|Drama              5.00
    It Happened One Night (1934)           Comedy|Romance              5.00
          His Girl Friday (1940)           Comedy|Romance              5.00
Streetcar Named Desire, A (1951)                    Drama              5.00
           Paths of Glory (1957)                Drama|War              5.00
                      Ran (1985)                Drama|War              5.00
            Touch of Evil (1958) Crime|Film-Noir|Thriller              5.00
             Hustler, The (1961)                    Drama              5.00
           Secrets & Lies (1996)                    Drama              4.99
         Five Easy Pieces (1970)                    Drama              4.99

Top 10 Funk SVD Recommendations for User 5
                                          title                              

### Step 7: Evaluate - RMSE, Precision@K, Recall@K

In [10]:
def precision_recall_at_k(model, train_data, test_data, movies_df, k=10, threshold=4.0):
    precisions, recalls = [], []
    for uid in test_data['userId'].unique():
        user_test = test_data[(test_data['userId'] == uid) & (test_data['rating'] >= threshold)]
        relevant = set(user_test['movieId'])
        if not relevant:
            continue
        recs = recommend_funk_svd(uid, model, train_data, movies_df, top_n=k)
        if recs.empty:
            continue
        rec_ids = set(recs.merge(movies_df, left_on='title', right_on='title')['movieId'])
        hits = rec_ids & relevant
        precisions.append(len(hits) / k)
        recalls.append(len(hits) / len(relevant))
    return np.mean(precisions) if precisions else 0, np.mean(recalls) if recalls else 0, len(precisions)

print('Computing Precision@10 and Recall@10 (this may take a minute)...')
prec, rec, n_users = precision_recall_at_k(model_tuned, train_df, test_df, movies, k=10)

print(f'\nFunk SVD (Tuned) Evaluation:')
print(f'  RMSE:         {rmse_tuned:.4f}')
print(f'  Precision@10: {prec:.4f}')
print(f'  Recall@10:    {rec:.4f}')
print(f'  Evaluated users: {n_users}')

Computing Precision@10 and Recall@10 (this may take a minute)...

Funk SVD (Tuned) Evaluation:
  RMSE:         0.8740
  Precision@10: 0.0132
  Recall@10:    0.0098
  Evaluated users: 599


### Step 8: Compare Funk SVD (Task 6) vs Manual Truncated SVD (Task 5)

In [11]:
from scipy.sparse.linalg import svds

# Manual Truncated SVD (Task 5 approach) on same split
train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
user_means = train_matrix.mean(axis=1)
train_centered = train_matrix.sub(user_means, axis=0).fillna(0)

U, sigma, Vt = svds(train_centered.values, k=50)
pred_manual = np.clip(U @ np.diag(sigma) @ Vt + user_means.values.reshape(-1, 1), 0.5, 5.0)
pred_manual_df = pd.DataFrame(pred_manual, index=train_matrix.index, columns=train_matrix.columns)

manual_actuals, manual_preds = [], []
for _, row in test_df.iterrows():
    uid, mid = int(row['userId']), int(row['movieId'])
    if uid in pred_manual_df.index and mid in pred_manual_df.columns:
        manual_actuals.append(row['rating'])
        manual_preds.append(pred_manual_df.loc[uid, mid])

manual_rmse = np.sqrt(mean_squared_error(manual_actuals, manual_preds))

print('=' * 60)
print('COMPARISON: Truncated SVD (Task 5) vs Funk SVD (Task 6)')
print('=' * 60)
print(f'{"Metric":<15s} | {"Truncated SVD":>14s} | {"Funk SVD":>14s}')
print('-' * 50)
print(f'{"RMSE":<15s} | {manual_rmse:>14.4f} | {rmse_tuned:>14.4f}')
print(f'{"Precision@10":<15s} | {"n/a":>14s} | {prec:>14.4f}')
print(f'{"Recall@10":<15s} | {"n/a":>14s} | {rec:>14.4f}')

COMPARISON: Truncated SVD (Task 5) vs Funk SVD (Task 6)
Metric          |  Truncated SVD |       Funk SVD
--------------------------------------------------
RMSE            |         0.9304 |         0.8740
Precision@10    |            n/a |         0.0132
Recall@10       |            n/a |         0.0098


### Analysis: Truncated SVD vs Funk SVD

Key differences between Task 5 (truncated SVD) and Task 6 (Funk SVD):

1. Bias modeling: Funk SVD explicitly models global mean, user bias, and item bias. Truncated SVD only handles user mean centering and misses item-level biases.

2. Missing value handling: Truncated SVD fills missing entries with 0 after centering, which introduces noise. Funk SVD only trains on observed ratings, avoiding this issue.

3. Regularization: Funk SVD applies L2 regularization to all parameters to prevent overfitting. Truncated SVD has no regularization.

4. Optimization: Truncated SVD computes an exact truncated decomposition in one shot. Funk SVD uses iterative SGD, optimizing specifically for rating prediction accuracy.

5. Scalability: Funk SVD scales better to large sparse matrices because it never materializes the full dense matrix. Truncated SVD requires the full dense matrix in memory.

Funk SVD (the approach used by the Surprise library SVD class) typically achieves better RMSE due to these design choices, especially the explicit bias terms and training only on observed ratings.